In [1]:
import os
import pickle
import gc
from tqdm import tqdm
import time
import polars as pl
import numpy as np
import shutil
import glob
from tienlen_libs import Card, Utils, ActionGenerator, NaturalWin
from tienlen_libs.constant import ComboType
import math
from collections import defaultdict
from preprocessing_data_3_player import create_polars_table_of_state_action_return_encode, determine_early_mid_end_game_stage, generate_list_state_action_return_encode,undersample_df_table_for_mid_end_game_stage 

In [2]:
def check_data(list_file_src_dir):
    list_who_play_first = []
    set_player1_card = set()
    for file_dir in tqdm(list_file_src_dir):
        try:
            with open(file_dir, 'rb') as f:
                data = pickle.load(f)
            list_who_play_first.append(data["who_play_first"])
            player1_card = tuple(data["player_card"][1])
            set_player1_card.add(player1_card)
            for state in data["Q_table"]:
                player_action = state[1]
                assert len(player_action) == 2
                player2_action = player_action[0]
                player3_action = player_action[1]
                turn = state[2]
                if turn == 1:
                    for player_id_action in player_action:
                        assert player_id_action == tuple() or player_id_action == -1
                elif turn == 2:
                    assert isinstance(player2_action, tuple) and len(player2_action) > 0
                    assert player3_action == tuple() or player3_action == -1
                elif turn == 3:
                    assert isinstance(player3_action, tuple) and len(player3_action) > 0

                player_card = state[0]
                for i in range(1, 4):
                    assert set(player_card[i-1]).issubset(set(data["player_card"][i]))
        except BaseException as e:
            print(f"ERROR with file {file_dir}")
            print(e)
    return list_who_play_first, set_player1_card

In [ ]:
file_data_dir = r"D:\Project\TLMN\data_see_opponent_card_3_players\TLMN_2025-06-10-00-22-27_CP23027HUGO_26436_325.pkl"
dest_dir = r"F:\tmp.parquet"

create_polars_table_of_state_action_return_encode([(file_data_dir, dest_dir)])

In [ ]:
tmp = pl.read_parquet(dest_dir)
for row in tmp.iter_rows(named=True):
    player2_card = []
    player3_card = []
    for i in range(52):
        if row[f"player2_card_{i}"]:
            player2_card.append(i)
        if row[f"player3_card_{i}"]:
            player3_card.append(i)
    print(f"len_player2_card: {len(player2_card)}, len_player3_card: {len(player3_card)}", end=": ")
    print(determine_early_mid_end_game_stage(([], player2_card, player3_card)))